# 03 — Fine-tuning IndoBERT untuk Analisis Sentimen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/06-transformers-nlp/03_finetune_indobert_sentimen.ipynb)

Sekarang kita memakai model **encoder** pretrained (keluarga BERT) untuk tugas nyata: **klasifikasi sentimen kalimat bahasa Indonesia**. Alurnya sama seperti transfer learning gambar (modul 02 dan 05): ambil model pretrained, tambah head klasifikasi, lalu fine-tune.

Isi notebook:
1. **Baseline klasik**: TF-IDF + Logistic Regression (ML dasar yang sudah kamu kuasai).
2. Fine-tune **IndoBERT** dengan `Trainer` dari Hugging Face.
3. Evaluasi: akurasi, **macro-F1**, confusion matrix, dan analisis kesalahan.
4. Inference dengan `pipeline` dan menyimpan model.

**Dataset: NusaX-Senti (bahasa Indonesia)**, kalimat ulasan/komentar berlabel `negative`, `neutral`, `positive` (500 train, 100 validasi, 400 test). Sumber: [IndoNLP/nusax](https://github.com/IndoNLP/nusax), lisensi data **CC-BY-SA**. Data diunduh langsung dari GitHub dan tidak disimpan di repo ini.

> Model: [`indobenchmark/indobert-base-p1`](https://huggingface.co/indobenchmark/indobert-base-p1) (IndoNLU), BERT-base yang di-pretrain pada korpus bahasa Indonesia. Periksa lisensi di halaman model sebelum dipakai di luar keperluan belajar.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi dan Konfigurasi

`datasets` dan `accelerate` (dibutuhkan `Trainer`) biasanya sudah ada di Colab; sel ini memastikannya.

In [ ]:
%pip install -q "transformers>=4.46" datasets accelerate scikit-learn pandas

Import dan konfigurasi. `QUICK_RUN=1` memakai subset kecil dan 1 epoch.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline)

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
MODEL_NAME = "indobenchmark/indobert-base-p1"
EPOCHS = 1 if QUICK_RUN else 5
NUSAX_URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment/indonesian"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print(f"device: {device} | model: {MODEL_NAME} | epochs: {EPOCHS}")

## 2. Data

Label teks diubah menjadi angka (`id2label` / `label2id`). Mapping ini nanti disimpan di konfigurasi model, sehingga prediksi langsung berupa nama label.

In [ ]:
dfs = {s: pd.read_csv(f"{NUSAX_URL}/{s}.csv") for s in ["train", "valid", "test"]}
LABELS = ["negative", "neutral", "positive"]
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
for df in dfs.values():
    df["label_id"] = df["label"].map(label2id)
if QUICK_RUN:
    dfs = {s: df.sample(min(len(df), 64), random_state=0) for s, df in dfs.items()}

for s, df in dfs.items():
    print(f"{s:<5} {len(df):>3} kalimat | {df['label'].value_counts().to_dict()}")
dfs["train"][["text", "label"]].head()

## 3. Baseline: TF-IDF + Logistic Regression

Selalu buat **baseline sederhana** dulu. Jika model besar tidak jauh lebih baik, kompleksitas tambahannya tidak sepadan.

TF-IDF mengubah teks menjadi vektor frekuensi kata (plus pasangan kata/*bigram*), lalu Logistic Regression mempelajari bobot tiap kata. Kelemahannya: tidak memahami urutan kata dan konteks (misalnya "tidak enak" vs "enak"), dan kata yang tidak ada di data train diabaikan.

Metrik utama: **macro-F1** (rata-rata F1 ketiga kelas). Metrik ini lebih jujur daripada akurasi ketika kelas tidak seimbang (kelas `neutral` paling sedikit).

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X_train = tfidf.fit_transform(dfs["train"]["text"])
X_test = tfidf.transform(dfs["test"]["text"])
baseline = LogisticRegression(max_iter=2000, C=5.0, class_weight="balanced").fit(X_train, dfs["train"]["label_id"])
pred_base = baseline.predict(X_test)

y_test = dfs["test"]["label_id"].to_numpy()
hasil = {"TF-IDF + LogReg": (accuracy_score(y_test, pred_base), f1_score(y_test, pred_base, average="macro"))}
print(f"Baseline → akurasi {hasil['TF-IDF + LogReg'][0]:.3f} | macro-F1 {hasil['TF-IDF + LogReg'][1]:.3f}")

# Kata dengan bobot terbesar per kelas: apa yang "dipelajari" baseline?
fitur = np.array(tfidf.get_feature_names_out())
for i, lab in id2label.items():
    print(f"{lab:<9}: {', '.join(fitur[np.argsort(baseline.coef_[i])[-8:][::-1]])}")

## 4. Tokenisasi untuk IndoBERT

Kita ubah DataFrame menjadi `datasets.Dataset` lalu tokenisasi semua kalimat dengan `map` (cepat, dan hasilnya di-*cache*).

**Padding tidak dilakukan di sini**, tetapi per batch oleh `DataCollatorWithPadding` (*dynamic padding*): setiap batch hanya dipadding sepanjang kalimat terpanjang di batch itu, bukan sepanjang `max_length`. Hasilnya training lebih cepat.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenisasi(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

ds = {s: Dataset.from_pandas(df[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)
      .map(tokenisasi, batched=True) for s, df in dfs.items()}
print(ds["train"])

panjang = [len(x) for x in ds["train"]["input_ids"]]
print(f"panjang token: rata-rata {np.mean(panjang):.0f}, maksimum {max(panjang)}")
collator = DataCollatorWithPadding(tokenizer)

## 5. Model dan Trainer

`AutoModelForSequenceClassification` = BERT + **head klasifikasi** di atas representasi token `[CLS]` (idenya sama dengan CLS token di ViT modul 05).

`Trainer` mengurus training loop yang selama ini kita tulis manual: batching, optimizer AdamW, scheduler, mixed precision (`fp16`), evaluasi per epoch, dan menyimpan checkpoint terbaik. Yang perlu kita sediakan hanyalah `compute_metrics`.

Hyperparameter yang umum untuk fine-tuning BERT: learning rate `2e-5` sampai `5e-5`, 3–5 epoch, dan warmup ~10%.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
print(f"Parameter: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} juta")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, pred), "macro_f1": f1_score(labels, pred, average="macro")}


BATCH = 16
total_steps = int(np.ceil(len(ds["train"]) / BATCH)) * EPOCHS
args = TrainingArguments(
    output_dir="indobert-sentimen",
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=max(1, int(0.1 * total_steps)),   # warmup 10% langkah pertama
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,            # kembalikan checkpoint dengan macro-F1 validasi terbaik
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),         # mixed precision (modul 05) jika ada GPU
    logging_steps=10,
    report_to="none",                       # tidak mengirim log ke layanan luar (wandb, dll.)
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["valid"],
                  processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics)

Jalankan training. Dengan 500 kalimat dan GPU T4, 5 epoch hanya butuh sekitar 1–2 menit.

In [ ]:
trainer.train()

log = pd.DataFrame(trainer.state.log_history)
ev = log.dropna(subset=["eval_macro_f1"])
print(ev[["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1"]].to_string(index=False))

## 6. Evaluasi di Data Test

Bandingkan dengan baseline. IndoBERT biasanya unggul jelas, terutama pada kalimat yang maknanya bergantung pada konteks dan negasi.

In [ ]:
pred_out = trainer.predict(ds["test"])
pred_bert = pred_out.predictions.argmax(-1)
hasil["IndoBERT fine-tune"] = (accuracy_score(y_test, pred_bert), f1_score(y_test, pred_bert, average="macro"))

print(f"{'model':<20} {'akurasi':>8} {'macro-F1':>9}")
for nama, (acc, f1) in hasil.items():
    print(f"{nama:<20} {acc:>8.3f} {f1:>9.3f}")
print()
print(classification_report(y_test, pred_bert, target_names=LABELS, digits=3, zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, pred, judul in [(ax[0], pred_base, "TF-IDF + LogReg"), (ax[1], pred_bert, "IndoBERT")]:
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=range(3)), display_labels=LABELS).plot(ax=a, colorbar=False)
    a.set_title(judul)
plt.tight_layout(); plt.show()

## 7. Analisis Kesalahan

Angka saja tidak cukup; baca kalimat yang salah diprediksi. Kita tampilkan kesalahan IndoBERT dengan **confidence tertinggi** (model paling yakin tapi salah). Sering ditemukan:
- label yang memang ambigu atau keliru di dataset,
- sarkasme,
- kalimat campuran (sebagian positif, sebagian negatif).

In [ ]:
probs = torch.softmax(torch.tensor(pred_out.predictions), dim=-1).numpy()
salah = dfs["test"].assign(pred=[id2label[i] for i in pred_bert], conf=probs.max(1))
salah = salah[salah["pred"] != salah["label"]].sort_values("conf", ascending=False)

print(f"{len(salah)} kesalahan dari {len(salah) + (pred_bert == y_test).sum()} kalimat\n")
for _, r in salah.head(8).iterrows():
    print(f"[asli: {r['label']:<8} | pred: {r['pred']:<8} ({r['conf']:.0%})] {r['text'][:140]}")

## 8. Inference dengan `pipeline` dan Menyimpan Model

`pipeline("text-classification")` membungkus tokenisasi, model, dan softmax menjadi satu fungsi yang mudah dipakai di aplikasi (modul 12: deployment).

In [ ]:
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1)
kalimat_baru = [
    "Aplikasinya sering error, saya kecewa sekali.",
    "Harga terjangkau dan pengirimannya cepat, mantap!",
    "Toko buka jam 9 pagi sampai jam 5 sore.",
    "Makanannya enak sih, tapi pelayanannya lama banget.",
]
for k, h in zip(kalimat_baru, clf(kalimat_baru)):
    print(f"{h['label']:<9} {h['score']:.0%}  {k}")

SAVE_DIR = "indobert-sentimen-final"
trainer.save_model(SAVE_DIR)          # menyimpan model + tokenizer + id2label
print("\nDisimpan:", sorted(os.listdir(SAVE_DIR)))

## Ringkasan

- Selalu mulai dengan **baseline** klasik (TF-IDF + Logistic Regression) sebagai pembanding.
- `AutoModelForSequenceClassification` = encoder pretrained + head klasifikasi di atas `[CLS]`; sertakan `id2label`/`label2id`.
- `Trainer` + `TrainingArguments` menggantikan training loop manual: evaluasi per epoch, simpan checkpoint terbaik, mixed precision.
- **Dynamic padding** (`DataCollatorWithPadding`) mempercepat training.
- Pakai **macro-F1** untuk kelas tidak seimbang, dan selalu baca contoh kesalahan.

## Latihan

1. **Model multibahasa vs khusus Indonesia.** Fine-tune `xlm-roberta-base` dengan pengaturan yang sama dan bandingkan macro-F1, waktu training, serta jumlah token rata-rata per kalimat dengan IndoBERT (hubungkan dengan fertility di notebook 01). Periksa lisensi kedua model.
2. **Kurva data.** Latih IndoBERT dan baseline TF-IDF dengan 50, 100, 250, dan 500 kalimat train (ambil secara stratified). Plot macro-F1 test terhadap jumlah data untuk kedua model. Di titik mana IndoBERT mulai unggul jelas, dan apa artinya untuk proyek dengan data berlabel sedikit?
3. **Uji ketahanan.** Buat 15 kalimat uji buatanmu sendiri yang menantang: negasi (`"tidak buruk"`), sarkasme, bahasa gaul/singkatan (`"mantul"`, `"gk enak"`), dan kalimat campuran. Bandingkan prediksi baseline dan IndoBERT, lalu tulis 3 temuan tentang kelemahan model dan ide perbaikannya (data tambahan, augmentasi, atau model lain).